# Script to write rainy season characteristics following Bombardi et al. (2019) 

## at the end, there is an error for  CESM2-WACCM  G6solar  2021-2050 : FIXED IT

there is a data issue. check the length of the data to fix it

In [1]:
debug=False

In [2]:
# import inspect
# import os
# import sys

# import numpy as np
# import xarray as xr

# import matplotlib as mpl
# import matplotlib.pyplot as plt

# import math

# import cftime
# from datetime import timedelta, date
# from pathlib import Path

# analysis_path = os.path.abspath("../20260112_Basic_Analysis")
# sys.path.append(analysis_path)
# import myfunctions as mf

# sys.path.append('./rainyseason_functions/')
# from rainyseason_B17_onset import rainyseason_B17_onset
# print(inspect.signature(rainyseason_B17_onset))

In [3]:
import os
import sys

import numpy as np
import xarray as xr

import matplotlib as mpl
import matplotlib.pyplot as plt

import math

import cftime
from datetime import timedelta, date
from pathlib import Path

analysis_path = os.path.abspath("../20260112_Basic_Analysis")
sys.path.append(analysis_path)
import myfunctions as mf

sys.path.append('./rainyseason_functions/')
from rainyseason_onset import rainyseason_onset
from rainyseason_B17_onset import rainyseason_B17_onset
from rainyseason_demise import rainyseason_demise
from rainyseason_B17_demise import rainyseason_B17_demise

import warnings
warnings.filterwarnings("ignore")

In [4]:
# =========================
# Base CEDA paths
# =========================

CEDA_BASE = Path("/badc/cmip6/data/CMIP6")

In [5]:
# ── Configuration ─────────────────────────────────────────────────────────────

MODELS = {
    "UKESM1-0-LL":   {"institution": "MOHC",         "ensemble": "r1i1p1f2", "grid": "gn"},
    "CNRM-ESM2-1":   {"institution": "CNRM-CERFACS", "ensemble": "r1i1p1f2", "grid": "gr"},
    "MPI-ESM1-2-LR": {"institution": "MPI-M",        "ensemble": "r1i1p1f1", "grid": "gn"},
    "CESM2-WACCM":   {"institution": "NCAR",         "ensemble": "r1i1p1f1", "grid": "gn"},
    "IPSL-CM6A-LR":  {"institution": "IPSL",         "ensemble": "r1i1p1f1", "grid": "gr"},
}

EXPERIMENTS = {
    "HIST":     {"project": "CMIP",        "scenario": "historical"},
    "SSP245":   {"project": "ScenarioMIP", "scenario": "ssp245"},
    "SSP585":   {"project": "ScenarioMIP", "scenario": "ssp585"},
    "G6solar":  {"project": "GeoMIP",      "scenario": "G6solar"},
    "G6sulfur": {"project": "GeoMIP",      "scenario": "G6sulfur"},
}

TIME_SLICES = {
    "HIST":     [("1985-01-01", "2014-12-30")],
    "SSP245":   [("2021-01-01", "2050-12-30"), ("2071-01-01", "2100-12-30")],
    "SSP585":   [("2021-01-01", "2050-12-30"), ("2071-01-01", "2100-12-30")],
    "G6solar":  [("2021-01-01", "2050-12-30"), ("2071-01-01", "2100-12-30")],
    "G6sulfur": [("2021-01-01", "2050-12-30"), ("2071-01-01", "2100-12-30")],
}

varname  = "pr"
missval  = -999.0
dper     = 25.0
npass    = 50
OUT_DIR  = Path("/gws/ssde/j25b/impose/bidyut/analysis_transient_data/rainyseason_output_files")
# OUT_DIR  = Path("./rainyseason_output_files")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Helper functions (same as developed step by step) ─────────────────────────

def Harmonics(coefa, coefb, hvar, tseries, nmodes, missval):
    mtot  = len(tseries)
    time  = np.arange(1, mtot + 1, 1.)
    tdata = tseries.copy()
    tdata[tseries == missval] = 0.
    svar  = sum((tdata - np.mean(tdata))**2) / (mtot - 1)
    nm    = nmodes
    if 2 * nm > mtot:
        nm = mtot // 2
    coefa = np.zeros(nm); coefb = np.zeros(nm); hvar = np.zeros(nm)
    for tt in range(nm):
        Ak = np.sum(tdata * np.cos(2. * math.pi * (tt+1) * time / float(mtot)))
        Bk = np.sum(tdata * np.sin(2. * math.pi * (tt+1) * time / float(mtot)))
        coefa[tt] = Ak * 2. / float(mtot)
        coefb[tt] = Bk * 2. / float(mtot)
        hvar[tt]  = mtot * (coefa[tt]**2 + coefb[tt]**2) / (2. * (mtot-1) * svar)
    return coefa, coefb, hvar

def build_time_arrays(time_coord):
    times = time_coord.values
    if isinstance(times[0], cftime.datetime):
        year  = np.array([t.year  for t in times], dtype=int)
        month = np.array([t.month for t in times], dtype=int)
        day   = np.array([t.day   for t in times], dtype=int)
        jday  = np.array([t.timetuple().tm_yday for t in times], dtype=int)
    else:
        import pandas as pd
        dti   = pd.DatetimeIndex(times)
        year  = dti.year.values.astype(int)
        month = dti.month.values.astype(int)
        day   = dti.day.values.astype(int)
        jday  = dti.day_of_year.values.astype(int)
    return jday, day, month, year

# This function was writing time wrongly due to some float overflow issue
# def make_var(data, long_name, units, nyrs_save):
#     return xr.DataArray(
#         data[:nyrs_save],
#         dims=["year", "lat", "lon"],
#         attrs={"long_name": long_name, "units": units, "missing_value": missval},
#     )

def make_var(data, long_name, units, nyrs_save):
    return xr.DataArray(
        data[:nyrs_save].astype(np.float32),   # ← explicit cast
        dims=["year", "lat", "lon"],
        attrs={"long_name": long_name, "units": units, "missing_value": missval},
    )

def julian(dd,mm,yy,noleap=1):
    """
    Function that calculates julian days [Day of Year] from day,month, and year imput
    Imput:
       yy:     year [integer]
       mm:     month [integer]
       dd:     day [integer]
       noleap = flag to indicate whether or not leap years should be considered.
       noleap = 0 --> time searies contain Feb 29
       noleap = 1 --> time series does not contain Feb 29
   Output:
       jday:   COrresponding Julian day or Day of Year [1,366]
    Example:
    --------
      >>> jday = julian(1,1,1980) # no leap years
      >>> jday = julian(1,1,1980,0)
    """
    mon=[31,28,31,30,31,30,31,31,30,31,30,31]
    if noleap==0:
       if yy % 4 == 0 and yy % 100 != 0 or yy % 400 == 0:
          mon=[31,29,31,30,31,30,31,31,30,31,30,31]
    if mm == 1:
       jday=dd
    if mm > 1:
       jday=sum(mon[0:mm-1])+dd
    return jday


def convert_360_to_365(da_360):
    """
    Convert a 360-day calendar DataArray to a 365-day calendar
    by inserting NaN days at the end of months that have 31 days
    and removing Feb 29-30.
    
    Input:  xarray DataArray with Datetime360Day time axis
    Output: numpy array (ntot_365, nlat, nlon) with NaN inserted
            and corresponding year, month, day, jday arrays
    """
    import pandas as pd
    
    times = da_360.time.values
    data  = da_360.values  # (ntot_360, nlat, nlon)
    
    years  = np.array([t.year  for t in times])
    months = np.array([t.month for t in times])
    days   = np.array([t.day   for t in times])
    
    unique_years = np.unique(years)
    yr0 = int(unique_years[0])
    
    # Months that need an extra day (31-day months in real calendar)
    add_day_after = {1, 3, 5, 7, 8, 10, 12}
    # Feb 29 and 30 in 360-day need to be removed
    
    new_data   = []
    new_year   = []
    new_month  = []
    new_day    = []
    
    for yr in unique_years:
        for mo in range(1, 13):
            # Get the 30 days of this month in 360-day
            idx = np.where((years == yr) & (months == mo))[0]
            mo_data = data[idx, :, :]   # shape (30, nlat, nlon)
            mo_days = days[idx]          # 1..30
            
            if mo == 2:
                # Keep only Feb 1-28, drop Feb 29-30
                keep = mo_days <= 28
                mo_data = mo_data[keep]
                mo_days = mo_days[keep]
                for d, row in zip(mo_days, mo_data):
                    new_data.append(row)
                    new_year.append(yr)
                    new_month.append(mo)
                    new_day.append(d)
            else:
                # Add all 30 days
                for d, row in zip(mo_days, mo_data):
                    new_data.append(row)
                    new_year.append(yr)
                    new_month.append(mo)
                    new_day.append(int(d))
                # Insert NaN day at end of 31-day months
                if mo in add_day_after:
                    new_data.append(np.full(data.shape[1:], np.nan))
                    new_year.append(yr)
                    new_month.append(mo)
                    new_day.append(31)
    
    new_data  = np.array(new_data)   # (ntot_365, nlat, nlon)
    new_year  = np.array(new_year,  dtype=int)
    new_month = np.array(new_month, dtype=int)
    new_day   = np.array(new_day,   dtype=int)
    
    # Compute jday using Bombardi's julian() function
    new_jday = np.zeros(len(new_year), dtype=int)
    for tt in range(len(new_year)):
        new_jday[tt] = julian(int(new_day[tt]), int(new_month[tt]), int(new_year[tt]))
    
    print(f"  360-day ntot: {data.shape[0]}")
    print(f"  365-day ntot: {new_data.shape[0]}")
    print(f"  Expected:     {len(unique_years) * 365}")
    
    return new_data, new_year, new_month, new_day, new_jday
    
# ── Main loop ─────────────────────────────────────────────────────────────────

for model_name, model_meta in MODELS.items():
    for exp, meta in EXPERIMENTS.items():
        for t_start, t_end in TIME_SLICES[exp]:

            # ── Output filename ───────────────────────────────────────────────
            yr_s = t_start[:4]; yr_e = t_end[:4]
            outfile = OUT_DIR / f"rainy_season_{model_name}_{exp}_{yr_s}-{yr_e}.nc"

            if outfile.exists():
                print(f"  SKIP {outfile.name} — already exists")
                continue

            print(f"\n{'='*60}")
            print(f"  {model_name}  {exp}  {yr_s}-{yr_e}")
            print(f"{'='*60}")

            # ── CESM2-WACCM ensemble override ─────────────────────────────────
            if model_name == "CESM2-WACCM":
                ensemble = "r1i1p1f2" if meta["scenario"] == "G6sulfur" else "r1i1p1f1"
            else:
                ensemble = model_meta["ensemble"]

            base = (
                CEDA_BASE
                / meta["project"]
                / model_meta["institution"]
                / model_name
                / meta["scenario"]
                / ensemble
                / "day"
                / varname
                / model_meta["grid"]
                / "latest"
            )

            # ── Load data ─────────────────────────────────────────────────────
            try:
                ds = mf.open_files(str(base))
                da = mf.read_var(ds, varname)
            except Exception as e:
                print(f"  ERROR loading: {e} — skipping")
                continue

            # ── Time slice ────────────────────────────────────────────────────
            try:
                if model_name == "CESM2-WACCM":
                    # NoLeap calendar needs cftime objects
                    da = da.sel(time=slice(
                        cftime.DatetimeNoLeap(int(yr_s), 1, 1),
                        cftime.DatetimeNoLeap(int(yr_e), 12, 30)
                    ))
                else:
                    da = da.sel(time=slice(t_start, t_end))

                # After slicing and before da.load()
                _, idx = np.unique(da.time.values, return_index=True)
                da = da.isel(time=idx).sortby("time")
                da = da.load()
            except Exception as e:
                print(f"  ERROR slicing time: {e} — skipping")
                continue

            if da.sizes["time"] == 0:
                print(f"  WARNING: empty time slice — skipping")
                continue

            # ── Step 1: unit conversion + shape ──────────────────────────────
            da_mm = (da * 86400.0).transpose("time", "lat", "lon")
            da_mm.attrs["units"] = "mm/day"
            lats  = da_mm.lat.values
            lons  = da_mm.lon.values

            # Detect calendar
            calendar = da_mm.time.values[0].__class__.__name__
            print(f"  Calendar class: {calendar}")
            
            if calendar == "Datetime360Day":
                print("  Converting 360-day to 365-day calendar...")
                prec, year, month, day, jday = convert_360_to_365(da_mm)
                prec[np.isnan(prec)] = 0.0   # inserted padding days → dry
                prec[prec < 0.] = 0.0        # any remaining negatives → dry
                ntot    = prec.shape[0]
                tot_int = 365
                tot     = float(tot_int)
                nlat    = prec.shape[1]
                nlon    = prec.shape[2]
                yr0     = int(year[0])
                nyrs    = int(year.max() - year.min()) + 1
            else:
                prec = da_mm.values.copy()
                jday, day, month, year = build_time_arrays(da_mm.time)
                id = np.where((month == 2) & (day == 29))[0]
                if len(id) > 0:
                    prec  = np.delete(prec,  id, axis=0)
                    year  = np.delete(year,  id, axis=0)
                    month = np.delete(month, id, axis=0)
                    day   = np.delete(day,   id, axis=0)
                    jday  = np.delete(jday,  id, axis=0)
                ntot    = prec.shape[0]
                tot_int = 365
                tot     = float(tot_int)
                nlat    = prec.shape[1]
                nlon    = prec.shape[2]
                prec[np.isnan(prec)] = missval   # fill values → missval
                prec[prec < 0.]      = missval   # negative values → missval
                yr0     = int(year[0])
                nyrs    = int(year.max() - year.min()) + 1

            
            # prec  = da_mm.values

            # ntot  = prec.shape[0]
            # nlat  = prec.shape[1]
            # nlon  = prec.shape[2]
            # lats  = da_mm.lat.values
            # lons  = da_mm.lon.values

            # # ── Step 2: date arrays ───────────────────────────────────────────
            # # jday, day, month, year = build_time_arrays(da_mm.time)
            # # tot_int = int(round(ntot / len(np.unique(year))))  # 360 or 365
            # # yr0     = int(year[0])
            # # nyrs    = int(year.max() - year.min()) + 1
            # # tot     = float(tot_int)

            # # Convert 360-day to 365-day
            # prec, year, month, day, jday = convert_360_to_365(da_mm)
            # ntot   = prec.shape[0]
            # tot    = 365
            # tot_int = 365
            # yr0     = int(year[0])
            # nyrs    = int(year.max() - year.min()) + 1
            
            # # prec[np.isnan(prec)] = missval
            # prec[np.isnan(prec)] = -1.0   # negative so excluded by tmp>=0, but not missval
            ###################################### DEBUG ################################################
            if debug:
                # Add this right after prec[np.isnan(prec)] = missval
                print(f"  ntot={ntot}, dper={dper}")
                print(f"  thres={ntot - 0.01*dper*ntot:.0f}")
                print(f"  valid days at [55,69]: {np.sum(prec[:,55,69] != missval)}")
            ###################################### DEBUG ################################################

            print(f"  Calendar: {type(da_mm.time.values[0]).__name__}  "
                  f"tot={tot_int}  ntot={ntot}  nlat={nlat}  nlon={nlon}")

            ###################################### DEBUG ################################################
            if debug:
                # Check jday values for January and December
                print("First 10 jdays:", jday[:10])
                print("Last 10 jdays:", jday[-10:])
                print("Max jday:", jday.max(), "Min jday:", jday.min())
                print("jday on Dec 1:", jday[np.where((month==12) & (day==1))[0][0]])

            # ── Step 3: mean annual cycle + harmonics + startwet ─────────────
            prec[prec < 0.] = 0.0 #missval
            thres = ntot - 0.01 * dper * ntot
            mask  = np.zeros((nlat, nlon))
            for it in range(nlat):
                for jt in range(nlon):
                    id = np.where(prec[:, it, jt] != missval)
                    if len(id[0]) >= thres:
                        mask[it, jt] = 1.

            rm = np.zeros((nlat, nlon))
            for it in range(nlat):
                for jt in range(nlon):
                    if mask[it, jt] == 1.:
                        tmp = prec[:, it, jt]
                        id  = np.where(tmp >= 0.)
                        if len(id[0]) > 1:
                            rm[it, jt] = np.mean(tmp[id[0]])

            ###################################### DEBUG ################################################
            if debug:
                print("Your jday[0], month[0], day[0], year[0]:", jday[0], month[0], day[0], year[0])
                print("Your jday[-1], month[-1], day[-1], year[-1]:", jday[-1], month[-1], day[-1], year[-1])
    
                print("Compare jday for Dec 1 and Dec 30")
                print("Your jday for Dec 1:", jday[np.where((month==12)&(day==1))[0][0]])
                print("Your jday for Dec 30:", jday[np.where((month==12)&(day==30))[0][0]])
                print("Your jday for Nov 30:", jday[np.where((month==11)&(day==30))[0][0]])
                
                print("prec min/max/mean:", prec[:,55,69].min(), prec[:,55,69].max(), prec[:,55,69].mean())
                print("prec[prec>0] mean:", prec[:,55,69][prec[:,55,69]>0].mean())
                print("Number of zeros:", np.sum(prec[:,55,69]==0.))
                print("Number of missval:", np.sum(prec[:,55,69]==missval))
    
                prec_au_Mine=prec[:,55,69]
                print('prec_au_Mine stored')
            ###################################### DEBUG ################################################

            cycle = np.zeros((tot_int, nlat, nlon))
            for tt in range(tot_int):
                id = np.where(jday == tt + 1)
                for it in range(nlat):
                    for jt in range(nlon):
                        if mask[it, jt] == 1.:
                            tmp = prec[id[0], it, jt]
                            id2 = np.where(tmp >= 0.)
                            if len(id2[0]) > 1:
                                cycle[tt, it, jt] = np.mean(tmp[id2[0]])

            if debug:
                cycle_au=cycle[:,55,69]
                print('cycle[:,55,69]', cycle[:,55,69])

            time_arr  = np.arange(1, tot_int + 1, 1.)
            harm1     = np.zeros((nlat, nlon))
            harm2     = np.zeros((nlat, nlon))
            harm3     = np.zeros((nlat, nlon))
            harmonic1 = np.zeros((tot_int, nlat, nlon))
            smoothed  = np.zeros((tot_int, nlat, nlon))

            for it in range(nlat):
                for jt in range(nlon):
                    if mask[it, jt] == 1.:
                        coefa = np.zeros(3); coefb = np.zeros(3); hvar = np.zeros(3)
                        coefa, coefb, hvar = Harmonics(coefa, coefb, hvar,
                                                       cycle[:, it, jt], 3, missval)
                        harm1[it, jt] = hvar[0]
                        harm2[it, jt] = hvar[1]
                        harm3[it, jt] = hvar[2]
                        harmonic1[:, it, jt] = rm[it, jt]
                        harmonic1[:, it, jt] += (
                            coefa[0] * np.cos(2.*math.pi*time_arr/tot)
                          + coefb[0] * np.sin(2.*math.pi*time_arr/tot))
                        # Full 3-harmonic smoothed cycle
                        smoothed[:, it, jt] = np.mean(cycle[:, it, jt])
                        for pp in range(3):
                            smoothed[:, it, jt] = (smoothed[:, it, jt]
                                + coefa[pp] * np.cos(2. * math.pi * time_arr * (pp+1) / float(tot_int))
                                + coefb[pp] * np.sin(2. * math.pi * time_arr * (pp+1) / float(tot_int)))                        

            id = np.where(harm2 >= harm1); mask[id] = 0.; rm[id] = 0.
            id = np.where(harm3 >= harm1); mask[id] = 0.; rm[id] = 0.

            startwet = np.zeros((nlat, nlon))
            for it in range(nlat):
                for jt in range(nlon):
                    if mask[it, jt] == 1.:
                        id = np.where(harmonic1[:, it, jt] == harmonic1[:, it, jt].min())
                        startwet[it, jt] = jday[id[0][0]]

            print(f"  Step 3 done. Active grid points: {int(mask.sum())}/{nlat*nlon}")


            # # --- FIX FOR SEASONS THAT CROSS CALENDAR YEAR (e.g. Australia) ---
            
            # # decide once, globally
            # shift_needed = False
            # shift_ref = None
            
            # for it in range(nlat):
            #     for jt in range(nlon):
            #         if startwet[it, jt] > tot_int * 0.5:
            #             shift_needed = True
            #             shift_ref = int(startwet[it, jt] - 1)
            #             break
            #     if shift_needed:
            #         break
            
            # # if shift_needed:
            # if shift_needed:
            #     print(f"  Applying hydrological year shift: {shift_ref} days")
            
            #     jday_adj = ((jday - shift_ref - 1) % tot_int) + 1
            #     year_adj = year.copy()
            #     year_adj[jday <= shift_ref] += 1
            
            #     jday = jday_adj.copy()
            #     year = year_adj.copy()
            
            #     # ── Trim to complete years only ───────────────────────────────────────
            #     days_per_year  = {y: np.sum(year == y) for y in np.unique(year)}
            #     complete_years = [y for y in np.sort(np.unique(year))
            #                       if days_per_year[y] == tot_int]
            #     complete_mask_arr = np.isin(year, complete_years)
            
            #     jday  = jday[complete_mask_arr]
            #     day   = day[complete_mask_arr]
            #     month = month[complete_mask_arr]
            #     year  = year[complete_mask_arr]
            #     prec  = prec[complete_mask_arr, :, :]
            #     ntot  = prec.shape[0]
            
            #     unique_years = np.sort(np.unique(year))
            #     yr0  = int(unique_years[0])
            #     nyrs = len(unique_years)
            #     print(f"  After trimming: yr0={yr0}, nyrs={nyrs}, ntot={ntot}")
            
            #     # ── Recompute mean annual cycle on shifted jday ───────────────────────
            #     cycle[:] = 0.
            #     for tt in range(tot_int):
            #         id_s = np.where(jday == tt + 1)
            #         for it in range(nlat):
            #             for jt in range(nlon):
            #                 if mask[it, jt] == 1.:
            #                     tmp = prec[id_s[0], it, jt]
            #                     id2 = np.where(tmp >= 0.)
            #                     if len(id2[0]) > 1:
            #                         cycle[tt, it, jt] = np.mean(tmp[id2[0]])
            
            #     # ── Recompute harmonics, harmonic1, smoothed, startwet ───────────────
            #     time_arr = np.arange(1, tot_int + 1, 1.)
            #     for it in range(nlat):
            #         for jt in range(nlon):
            #             if mask[it, jt] == 1.:
            #                 coefa = np.zeros(3); coefb = np.zeros(3); hvar = np.zeros(3)
            #                 coefa, coefb, hvar = Harmonics(coefa, coefb, hvar,
            #                                                cycle[:, it, jt], 3, missval)
            #                 harm1[it, jt] = hvar[0]
            #                 harm2[it, jt] = hvar[1]
            #                 harm3[it, jt] = hvar[2]
            #                 harmonic1[:, it, jt] = rm[it, jt]
            #                 harmonic1[:, it, jt] += (
            #                     coefa[0] * np.cos(2.*math.pi*time_arr/tot)
            #                   + coefb[0] * np.sin(2.*math.pi*time_arr/tot))
            #                 smoothed[:, it, jt] = np.mean(cycle[:, it, jt])
            #                 for pp in range(3):
            #                     smoothed[:, it, jt] += (
            #                         coefa[pp] * np.cos(2.*math.pi*time_arr*(pp+1)/float(tot_int))
            #                       + coefb[pp] * np.sin(2.*math.pi*time_arr*(pp+1)/float(tot_int)))
            
            #     # Re-apply harmonic dominance mask
            #     id = np.where(harm2 >= harm1); mask[id] = 0.; rm[id] = 0.
            #     id = np.where(harm3 >= harm1); mask[id] = 0.; rm[id] = 0.
            
            #     # Recompute startwet on shifted harmonic1
            #     startwet[:] = 0.
            #     for it in range(nlat):
            #         for jt in range(nlon):
            #             if mask[it, jt] == 1.:
            #                 id_min = np.where(harmonic1[:, it, jt] ==
            #                                   harmonic1[:, it, jt].min())
            #                 startwet[it, jt] = id_min[0][0] + 1  # +1 because jday is 1-based
            
            # print("Unique jday range:", jday.min(), jday.max())
            # print("Year min/max:", year.min(), year.max())
            
            # # ── RECOMPUTE nyrs/yr0 also for the no-shift case ────────────────────────────
            # unique_years = np.sort(np.unique(year))
            # nyrs = min(len(unique_years), ntot // tot_int)
            # yr0  = int(unique_years[0])
            # print(f"  Seasons after hydro shift: {nyrs}")


            # ── Step 4: onset and demise ──────────────────────────────────────
            onset_jday   = np.zeros((nyrs, nlat, nlon))
            onset_day    = np.zeros((nyrs, nlat, nlon))
            onset_month  = np.zeros((nyrs, nlat, nlon))
            onset_year   = np.zeros((nyrs, nlat, nlon))
            demise_jday  = np.zeros((nyrs, nlat, nlon))
            demise_day   = np.zeros((nyrs, nlat, nlon))
            demise_month = np.zeros((nyrs, nlat, nlon))
            demise_year  = np.zeros((nyrs, nlat, nlon))

            wjd = np.zeros(nyrs); wd = np.zeros(nyrs)
            wm  = np.zeros(nyrs); wy = np.zeros(nyrs)
            wsc = np.zeros((nyrs, tot_int // 2))   # (nyrs + 1) buffer of +1 is implemented to handle yt=30 due to LeapYear
            djd = np.zeros(nyrs); dd = np.zeros(nyrs)
            dm  = np.zeros(nyrs); dy = np.zeros(nyrs)
            dsc = np.zeros((nyrs, tot_int // 2))   # (nyrs + 1) buffer of +1 is implemented to handle yt=30 due to LeapYear
            ap  = np.zeros(ntot)

            prec[prec < 0.] = 0.   # VERY IMPORTANT


            ###################################### DEBUG ################################################
            if debug:
                it, jt = 55, 69  #  Australia debug point
                print("harmonic1 min index:", np.argmin(harmonic1[:, it, jt]))
                print("harmonic1 min value:", harmonic1[:, it, jt].min())
                print("rm:", rm[it, jt])
                print("harm1/harm2/harm3:", harm1[it,jt], harm2[it,jt], harm3[it,jt])
                
                # Run this BEFORE the main loop to find indices
                lat_target = -20.0 #30.0   # North America
                lon_target = 130.0 #260.0 # Australia
                it_dbg = np.argmin(np.abs(lats - lat_target))
                jt_dbg = np.argmin(np.abs(lons - lon_target))
                print(f"Debug point: it={it_dbg}, jt={jt_dbg}, lat={lats[it_dbg]:.2f}, lon={lons[jt_dbg]:.2f}")
                
                # After startwet is computed, before Step 4
                ilat_au = np.where((lats >= -21.0) & (lats <= -9.5))[0]
                ilon_au = np.where((lons >= 120.5) & (lons <= 149.0))[0]
                sw_au = startwet[ilat_au, :][:, ilon_au]
                sw_au_valid = sw_au[sw_au > 0.]
                print(f"Australia startwet: min={sw_au_valid.min():.0f}, max={sw_au_valid.max():.0f}, mean={sw_au_valid.mean():.1f}")
                print(f"Sample values: {sw_au_valid[:10]}")
            #############################################################################################

            print("  Step 4: onset/demise loop...")
            #############################################################################################
            # # DIAGNOSTIC - print for first active grid point
            # first_active_printed = False
            #############################################################################################
            for it in range(nlat):
                if it % 20 == 0:
                    print(f"    lat {it+1}/{nlat}")
                for jt in range(nlon):
                    if rm[it, jt] > 0.:

                        #############################################################################################
                        # # DIAGNOSTIC - print first active point only
                        # if not first_active_printed:
                        #     print(f"  First active grid point: it={it}, jt={jt}, lat={lats[it]:.2f}, lon={lons[jt]:.2f}")
                        #     first_active_printed_2 = True
                        #############################################################################################

                        
                        sdate  = startwet[it, jt]
                        ap[:]  = prec[:, it, jt] - rm[it, jt]
                        # Change implemented to handle yt=30 due to LeapYear
                        sid    = np.where(jday == sdate)[0]
                        sjday_ = jday[sid]; sday_  = day[sid]
                        smonth_= month[sid]; syear_ = year[sid]
                        # # Both occurrences of this block:
                        # sid     = np.where(jday == sdate)[0]
                        # sjday_  = np.append(jday[sid], 0)
                        # sday_   = np.append(day[sid], 0)
                        # smonth_ = np.append(month[sid], 0)
                        # syear_  = np.append(year[sid], 0)

                        # ONSET first pass
                        wjd[:]=0.; wd[:]=0.; wm[:]=0.; wy[:]=0.; wsc[:]=0.
                        wjd[:],wd[:],wm[:],wy[:],wsc[:,:] = rainyseason_onset(
                            nyrs, tot_int, jday, day, month, year,
                            sdate, ap, wjd, wd, wm, wy, wsc)
                        miss = np.where(wjd==0.); id = np.where(wjd!=0.)
                        if len(id[0]) > 0:
                            tmpx = np.cos(wjd*math.pi/183.); tmpy = np.sin(wjd*math.pi/183.)
                            med  = math.atan2(np.median(tmpy[id]),np.median(tmpx[id]))*183./math.pi
                            if med < 0.: med += tot
                            tmpc = wjd[:] - med
                            if len(miss[0]) > 0: tmpc[miss] = 0.
                            pos = np.where(tmpc > tot*0.5);  tmpc[pos] -= tot
                            neg = np.where(tmpc < -tot*0.5); tmpc[neg] += tot
                            iqr  = np.percentile(tmpc[id],75) - np.percentile(tmpc[id],25)
                            outl = np.where(np.abs(tmpc) > iqr*1.5)
                            if len(outl[0]) > 0:
                                wjd[outl]=0.; wd[outl]=0.; wm[outl]=0.; wy[outl]=0.
                            onset_jday[:,it,jt]=wjd; onset_day[:,it,jt]=wd
                            onset_month[:,it,jt]=wm; onset_year[:,it,jt]=wy

                            # ONSET second pass — match Bombardi exactly
                            outl = np.where(wjd==0.)
                            if len(outl[0]) > 0:
                                wjd[:]=0.; wd[:]=0.; wm[:]=0.; wy[:]=0.
                                # Change implemented to handle yt=30 due to LeapYear
                                sid    = np.where(jday == sdate)[0]
                                sjday_ = jday[sid]; sday_  = day[sid]
                                smonth_= month[sid]; syear_ = year[sid]
                                _wjd, _wd, _wm, _wy = rainyseason_B17_onset(
                                    nyrs, tot_int, jday, day, month, year,
                                    sdate, ap, npass, wjd, wd, wm, wy)  # ← pass wjd,wd,wm,wy not sjday_
                                n = min(len(_wjd), nyrs)
                                wjd[:n]=_wjd[:n]; wd[:n]=_wd[:n]; wm[:n]=_wm[:n]; wy[:n]=_wy[:n]
                                onset_jday[outl,it,jt]=wjd[outl]; onset_day[outl,it,jt]=wd[outl]
                                onset_month[outl,it,jt]=wm[outl]; onset_year[outl,it,jt]=wy[outl]

                            # # ONSET second pass — match original signature exactly
                            # outl = np.where(wjd==0.)
                            # if len(outl[0]) > 0:
                            #     wjd[:]=0.; wd[:]=0.; wm[:]=0.; wy[:]=0.
                            #     sid     = np.where(jday == sdate)[0]
                            #     sjday_  = jday[sid];  sday_   = day[sid]
                            #     smonth_ = month[sid]; syear_  = year[sid]
                            #     _wjd, _wd, _wm, _wy = rainyseason_B17_onset(
                            #         nyrs, tot_int, jday, day, month, year,
                            #         sdate, ap, npass, sjday_, sday_, smonth_, syear_)
                            #     n = min(len(_wjd), nyrs)
                            #     wjd[:n]=_wjd[:n]; wd[:n]=_wd[:n]; wm[:n]=_wm[:n]; wy[:n]=_wy[:n]
                            #     onset_jday[outl,it,jt]=wjd[outl]; onset_day[outl,it,jt]=wd[outl]
                            #     onset_month[outl,it,jt]=wm[outl]; onset_year[outl,it,jt]=wy[outl]

                            # # ONSET second pass
                            # outl = np.where(wjd==0.)
                            # if len(outl[0]) > 0:
                            #     wjd[:]=0.; wd[:]=0.; wm[:]=0.; wy[:]=0.
                            #     # Change implemented to handle yt=30 due to LeapYear
                            #     sid     = np.where(jday == sdate)[0]
                            #     sjday_  = jday[sid];  sday_   = day[sid]
                            #     smonth_ = month[sid]; syear_  = year[sid]
                            #     # # Both occurrences of this block:
                            #     # sid     = np.where(jday == sdate)[0]
                            #     # sjday_  = np.append(jday[sid], 0)
                            #     # sday_   = np.append(day[sid], 0)
                            #     # smonth_ = np.append(month[sid], 0)
                            #     # syear_  = np.append(year[sid], 0)
                            #     _wjd, _wd, _wm, _wy = rainyseason_B17_onset(
                            #         nyrs, tot_int, jday, day, month, year,
                            #         sdate, ap, npass, sjday_, sday_, smonth_, syear_)
                            #     n = min(len(_wjd), nyrs)
                            #     wjd[:n]=_wjd[:n]; wd[:n]=_wd[:n]; wm[:n]=_wm[:n]; wy[:n]=_wy[:n]
                            #     onset_jday[outl,it,jt]=wjd[outl]; onset_day[outl,it,jt]=wd[outl]
                            #     onset_month[outl,it,jt]=wm[outl]; onset_year[outl,it,jt]=wy[outl]
                                
                            # # # ONSET second pass
                            # # outl = np.where(wjd==0.)
                            # # if len(outl[0]) > 0:
                            # #     wjd[:]=0.; wd[:]=0.; wm[:]=0.; wy[:]=0.
                            # #     wjd[:],wd[:],wm[:],wy[:] = rainyseason_B17_onset(
                            # #         nyrs, tot_int, jday, day, month, year,
                            # #         sdate, ap, npass, sjday_, sday_, smonth_, syear_)
                            # #     wjd[:n]=_wjd[:n]; wd[:n]=_wd[:n]; wm[:n]=_wm[:n]; wy[:n]=_wy[:n]
                            # #     onset_jday[outl,it,jt]=wjd[outl]; onset_day[outl,it,jt]=wd[outl]
                            # #     onset_month[outl,it,jt]=wm[outl]; onset_year[outl,it,jt]=wy[outl]

                                
                                wjd[:] = onset_jday[:,it,jt]
                                miss = np.where(wjd==0.); id = np.where(wjd!=0.)
                                if len(id[0]) > 0:
                                    tmpx = np.cos(wjd*math.pi/183.); tmpy = np.sin(wjd*math.pi/183.)
                                    med  = math.atan2(np.median(tmpy[id]),np.median(tmpx[id]))*183./math.pi
                                    if med < 0.: med += tot
                                    tmpc = wjd[:] - med
                                    if len(miss[0]) > 0: tmpc[miss] = 0.
                                    pos = np.where(tmpc > tot*0.5);  tmpc[pos] -= tot
                                    neg = np.where(tmpc < -tot*0.5); tmpc[neg] += tot
                                    iqr  = np.percentile(tmpc[id],75) - np.percentile(tmpc[id],25)
                                    outl = np.where(np.abs(tmpc) > iqr*3.)
                                    if len(outl[0]) > 0:
                                        onset_jday[outl,it,jt]=0.; onset_day[outl,it,jt]=0.
                                        onset_month[outl,it,jt]=0.; onset_year[outl,it,jt]=0.

                        # DEMISE first pass
                        djd[:]=0.; dd[:]=0.; dm[:]=0.; dy[:]=0.; dsc[:]=0.
                        djd[:],dd[:],dm[:],dy[:],dsc[:,:] = rainyseason_demise(
                            nyrs, tot_int, jday, day, month, year,
                            sdate, ap, djd, dd, dm, dy, dsc)
                        miss = np.where(djd==0.); id = np.where(djd!=0.)
                        if len(id[0]) > 0:
                            tmpx = np.cos(djd*math.pi/183.); tmpy = np.sin(djd*math.pi/183.)
                            med  = math.atan2(np.median(tmpy[id]),np.median(tmpx[id]))*183./math.pi
                            if med < 0.: med += tot
                            tmpc = djd[:] - med
                            if len(miss[0]) > 0: tmpc[miss] = 0.
                            pos = np.where(tmpc > tot*0.5);  tmpc[pos] -= tot
                            neg = np.where(tmpc < -tot*0.5); tmpc[neg] += tot
                            iqr  = np.percentile(tmpc[id],75) - np.percentile(tmpc[id],25)
                            outl = np.where(np.abs(tmpc) > iqr*1.5)
                            if len(outl[0]) > 0:
                                djd[outl]=0.; dd[outl]=0.; dm[outl]=0.; dy[outl]=0.
                            demise_jday[:,it,jt]=djd; demise_day[:,it,jt]=dd
                            demise_month[:,it,jt]=dm; demise_year[:,it,jt]=dy

                            # DEMISE second pass — match Bombardi exactly
                            outl = np.where(djd==0.)
                            if len(outl[0]) > 0:
                                djd[:]=0.; dd[:]=0.; dm[:]=0.; dy[:]=0.
                                _djd, _dd, _dm, _dy = rainyseason_B17_demise(
                                    nyrs, tot_int, jday, day, month, year,
                                    sdate, ap, npass, djd, dd, dm, dy)  # ← pass djd,dd,dm,dy not sjday_
                                n = min(len(_djd), nyrs)
                                djd[:n]=_djd[:n]; dd[:n]=_dd[:n]; dm[:n]=_dm[:n]; dy[:n]=_dy[:n]
                                demise_jday[outl,it,jt]=djd[outl]; demise_day[outl,it,jt]=dd[outl]
                                demise_month[outl,it,jt]=dm[outl]; demise_year[outl,it,jt]=dy[outl]
                                
                            # # DEMISE second pass
                            # outl = np.where(djd==0.)
                            # if len(outl[0]) > 0:
                            #     djd[:]=0.; dd[:]=0.; dm[:]=0.; dy[:]=0.
                            #     sid     = np.where(jday == sdate)[0]
                            #     sjday_  = jday[sid];  sday_   = day[sid]
                            #     smonth_ = month[sid]; syear_  = year[sid]
                            #     _djd, _dd, _dm, _dy = rainyseason_B17_demise(
                            #         nyrs, tot_int, jday, day, month, year,
                            #         sdate, ap, npass, sjday_, sday_, smonth_, syear_)
                            #     n = min(len(_djd), nyrs)
                            #     djd[:n]=_djd[:n]; dd[:n]=_dd[:n]; dm[:n]=_dm[:n]; dy[:n]=_dy[:n]
                            #     demise_jday[outl,it,jt]=djd[outl]; demise_day[outl,it,jt]=dd[outl]
                            #     demise_month[outl,it,jt]=dm[outl]; demise_year[outl,it,jt]=dy[outl]
                            # # # DEMISE second pass
                            # # outl = np.where(djd==0.)
                            # # if len(outl[0]) > 0:
                            # #     djd[:]=0.; dd[:]=0.; dm[:]=0.; dy[:]=0.
                            # #     djd[:],dd[:],dm[:],dy[:] = rainyseason_B17_demise(
                            # #         nyrs, tot_int, jday, day, month, year,
                            # #         sdate, ap, npass, sjday_, sday_, smonth_, syear_)
                            # #     demise_jday[outl,it,jt]=djd[outl]; demise_day[outl,it,jt]=dd[outl]
                            # #     demise_month[outl,it,jt]=dm[outl]; demise_year[outl,it,jt]=dy[outl]
                                djd[:] = demise_jday[:,it,jt]
                                miss = np.where(djd==0.); id = np.where(djd!=0.)
                                if len(id[0]) > 0:
                                    tmpx = np.cos(djd*math.pi/183.); tmpy = np.sin(djd*math.pi/183.)
                                    med  = math.atan2(np.median(tmpy[id]),np.median(tmpx[id]))*183./math.pi
                                    if med < 0.: med += tot
                                    tmpc = djd[:] - med
                                    if len(miss[0]) > 0: tmpc[miss] = 0.
                                    pos = np.where(tmpc > tot*0.5);  tmpc[pos] -= tot
                                    neg = np.where(tmpc < -tot*0.5); tmpc[neg] += tot
                                    iqr  = np.percentile(tmpc[id],75) - np.percentile(tmpc[id],25)
                                    outl = np.where(np.abs(tmpc) > iqr*3.)
                                    if len(outl[0]) > 0:
                                        demise_jday[outl,it,jt]=0.; demise_day[outl,it,jt]=0.
                                        demise_month[outl,it,jt]=0.; demise_year[outl,it,jt]=0.

                        # 33% missing mask
                        if len(np.where(onset_jday[:,it,jt]==0.)[0])/float(nyrs) > 0.33:
                            rm[it,jt]=0.
                            onset_jday[:,it,jt]=0.; onset_day[:,it,jt]=0.
                            onset_month[:,it,jt]=0.; onset_year[:,it,jt]=0.
                        if len(np.where(demise_jday[:,it,jt]==0.)[0])/float(nyrs) > 0.33:
                            rm[it,jt]=0.
                            demise_jday[:,it,jt]=0.; demise_day[:,it,jt]=0.
                            demise_month[:,it,jt]=0.; demise_year[:,it,jt]=0.


                        #######################################################################################
                        # # DIAGNOSTIC — add temporarily before the demise rearrangement
                        # if it == 0 and jt == 0:  # just one grid point
                        # # DIAGNOSTIC
                        # if not first_active_printed_2 and it==first_it and jt==first_jt:
                        #     print(f"  yr0={yr0}")
                        #     for yt in range(min(4, nyrs)):
                        #         print(f"  yt={yt}: onset_year={onset_year[yt,it,jt]:.0f} onset_jday={onset_jday[yt,it,jt]:.0f}"
                        #               f"  demise_year={demise_year[yt,it,jt]:.0f} demise_jday={demise_jday[yt,it,jt]:.0f}")

                        if debug:
                            if it == it_dbg and jt == jt_dbg:
                                print(f"  yr0={yr0}, rm={rm[it,jt]:.3f}")
                                for yt in range(min(4, nyrs)):
                                    print(f"  yt={yt}: onset_year={onset_year[yt,it,jt]:.0f} "
                                          f"onset_jday={onset_jday[yt,it,jt]:.0f}  "
                                          f"demise_year={demise_year[yt,it,jt]:.0f} "
                                          f"demise_jday={demise_jday[yt,it,jt]:.0f}")
                        #######################################################################################
        
                        # Demise year rearrangement
                        if demise_year[1,it,jt]==float(yr0) or demise_year[2,it,jt]==float(yr0+1):
                            # if demise_year[0,it,jt] < onset_year[0,it,jt]:
                            demise_year[0:nyrs-1,it,jt]  = demise_year[1:nyrs,it,jt];  demise_year[nyrs-1,it,jt]=0.
                            demise_month[0:nyrs-1,it,jt] = demise_month[1:nyrs,it,jt]; demise_month[nyrs-1,it,jt]=0.
                            demise_day[0:nyrs-1,it,jt]   = demise_day[1:nyrs,it,jt];   demise_day[nyrs-1,it,jt]=0.
                            demise_jday[0:nyrs-1,it,jt]  = demise_jday[1:nyrs,it,jt];  demise_jday[nyrs-1,it,jt]=0.
                        
                        # # ── Demise/onset year consistency check ──────
                        # for yt in range(nyrs - 1):
                        #     o_yr = onset_year[yt, it, jt]
                        #     d_yr = demise_year[yt, it, jt]
                        #     if o_yr == 0. or d_yr == 0.:
                        #         continue
                        #     # demise should be in the same or next year as onset
                        #     if d_yr < o_yr:
                        #         # shift demise forward by one slot
                        #         demise_jday[yt, it, jt]  = demise_jday[yt+1, it, jt]
                        #         demise_day[yt, it, jt]   = demise_day[yt+1, it, jt]
                        #         demise_month[yt, it, jt] = demise_month[yt+1, it, jt]
                        #         demise_year[yt, it, jt]  = demise_year[yt+1, it, jt]
                        # # zero out the last year which is now unreliable
                        # demise_jday[nyrs-1, it, jt]  = 0.
                        # demise_day[nyrs-1, it, jt]   = 0.
                        # demise_month[nyrs-1, it, jt] = 0.
                        # demise_year[nyrs-1, it, jt]  = 0.

            print("  Step 4 done.")

            # ── Step 5a: duration and accumulation ───────────────────────────
            durwet = np.zeros((nyrs, nlat, nlon))
            durdry = np.zeros((nyrs, nlat, nlon))
            totwet = np.zeros((nyrs, nlat, nlon))
            totdry = np.zeros((nyrs, nlat, nlon))

            for it in range(nlat):
                for jt in range(nlon):
                    for yt in range(nyrs):
                        if onset_year[yt,it,jt]==0. or demise_year[yt,it,jt]==0.:
                            continue
                        if demise_year[yt,it,jt] == onset_year[yt,it,jt]:
                            if demise_jday[yt,it,jt] < onset_jday[yt,it,jt]:
                                beg = int((demise_year[yt,it,jt]-yr0)*tot_int + demise_jday[yt,it,jt]-1)
                                ned = int((onset_year[yt,it,jt] -yr0)*tot_int + onset_jday[yt,it,jt] -1)
                                if 0 <= beg < ned <= ntot:
                                    durdry[yt,it,jt]=float(ned-beg); totdry[yt,it,jt]=np.sum(prec[beg:ned,it,jt])
                                if yt < nyrs-1 and demise_year[yt+1,it,jt] > 0.:
                                    beg = int((onset_year[yt,it,jt]   -yr0)*tot_int + onset_jday[yt,it,jt]   -1)
                                    ned = int((demise_year[yt+1,it,jt]-yr0)*tot_int + demise_jday[yt+1,it,jt]-1)
                                    if 0 <= beg < ned <= ntot:
                                        durwet[yt,it,jt]=float(ned-beg); totwet[yt,it,jt]=np.sum(prec[beg:ned,it,jt])
                            elif onset_jday[yt,it,jt] < demise_jday[yt,it,jt]:
                                beg = int((onset_year[yt,it,jt] -yr0)*tot_int + onset_jday[yt,it,jt] -1)
                                ned = int((demise_year[yt,it,jt]-yr0)*tot_int + demise_jday[yt,it,jt]-1)
                                if 0 <= beg < ned <= ntot:
                                    durwet[yt,it,jt]=float(ned-beg); totwet[yt,it,jt]=np.sum(prec[beg:ned,it,jt])
                                if yt < nyrs-1 and onset_year[yt+1,it,jt] > 0.:
                                    beg = int((demise_year[yt,it,jt]  -yr0)*tot_int + demise_jday[yt,it,jt]  -1)
                                    ned = int((onset_year[yt+1,it,jt] -yr0)*tot_int + onset_jday[yt+1,it,jt] -1)
                                    if 0 <= beg < ned <= ntot:
                                        durdry[yt,it,jt]=float(ned-beg); totdry[yt,it,jt]=np.sum(prec[beg:ned,it,jt])
                        elif 0. < demise_year[yt,it,jt] < onset_year[yt,it,jt]:
                            beg = int((demise_year[yt,it,jt]-yr0)*tot_int + demise_jday[yt,it,jt]-1)
                            ned = int((onset_year[yt,it,jt] -yr0)*tot_int + onset_jday[yt,it,jt] -1)
                            if 0 <= beg < ned <= ntot:
                                durdry[yt,it,jt]=float(ned-beg); totdry[yt,it,jt]=np.sum(prec[beg:ned,it,jt])
                            if yt < nyrs-1 and demise_year[yt+1,it,jt] > 0.:
                                if onset_jday[yt,it,jt] < demise_jday[yt+1,it,jt]:
                                    beg = int((onset_year[yt,it,jt]   -yr0)*tot_int + onset_jday[yt,it,jt]   -1)
                                    ned = int((demise_year[yt+1,it,jt]-yr0)*tot_int + demise_jday[yt+1,it,jt]-1)
                                    if 0 <= beg < ned <= ntot:
                                        durwet[yt,it,jt]=float(ned-beg); totwet[yt,it,jt]=np.sum(prec[beg:ned,it,jt])
                        elif 0. < onset_year[yt,it,jt] < demise_year[yt,it,jt]:
                            beg = int((onset_year[yt,it,jt] -yr0)*tot_int + onset_jday[yt,it,jt] -1)
                            ned = int((demise_year[yt,it,jt]-yr0)*tot_int + demise_jday[yt,it,jt]-1)
                            if 0 <= beg < ned <= ntot:
                                durwet[yt,it,jt]=float(ned-beg); totwet[yt,it,jt]=np.sum(prec[beg:ned,it,jt])
                            if yt < nyrs-1 and onset_year[yt+1,it,jt] > 0.:
                                if demise_jday[yt,it,jt] < onset_jday[yt+1,it,jt]:
                                    beg = int((demise_year[yt,it,jt]  -yr0)*tot_int + demise_jday[yt,it,jt]  -1)
                                    ned = int((onset_year[yt+1,it,jt] -yr0)*tot_int + onset_jday[yt+1,it,jt] -1)
                                    if 0 <= beg < ned <= ntot:
                                        durdry[yt,it,jt]=float(ned-beg); totdry[yt,it,jt]=np.sum(prec[beg:ned,it,jt])

            # Cap impossible durations
            for arr in [durwet, durdry]:
                arr[arr > tot_int] = 0.
            totwet[durwet == 0.] = 0.
            totdry[durdry == 0.] = 0.

            print("  Step 5a done.")

            # # ── Force float before saving ─────────────────────────────────────────────
            # for arr in [durwet, durdry, totwet, totdry,
            #             onset_jday, onset_day, onset_month, onset_year,
            #             demise_jday, demise_day, demise_month, demise_year]:
            #     arr = arr.astype(np.float64)

            # ── Step 5b: zero → missval ───────────────────────────────────────
            yrs_coord = np.arange(yr0, yr0 + nyrs, 1)
            nyrs_save = nyrs - 2    # drop last 2 years
            yr_start_coord = str(yrs_coord[0])
            yr_end_coord   = str(yrs_coord[nyrs_save - 1])

            # for arr in [onset_jday, onset_day, onset_month, onset_year,
            #             demise_jday, demise_day, demise_month, demise_year,
            #             totwet, totdry, durwet, durdry]:
            #     arr[arr == 0.] = missval

            # ── Step 5b: zero → missval, applied to actual arrays ─────────
            onset_jday[onset_jday   == 0.] = missval
            onset_day[onset_day     == 0.] = missval
            onset_month[onset_month == 0.] = missval
            onset_year[onset_year   == 0.] = missval
            demise_jday[demise_jday   == 0.] = missval
            demise_day[demise_day     == 0.] = missval
            demise_month[demise_month == 0.] = missval
            demise_year[demise_year   == 0.] = missval
            totwet[totwet == 0.] = missval
            totdry[totdry == 0.] = missval
            durwet[durwet == 0.] = missval
            durdry[durdry == 0.] = missval

            # ── Step 5c: save ─────────────────────────────────────────────────
            coords = {
                "year": yrs_coord[:nyrs_save],
                "doy":  np.arange(1, tot_int + 1),   # ← add this
                "lat":  lats,
                "lon":  lons,
            }

            ds_out = xr.Dataset(
                {
                    "onset_jday":   make_var(onset_jday,   "Wet season onset (Day of Year)",        "day_of_year", nyrs_save),
                    "onset_day":    make_var(onset_day,    "Wet season onset (day of month)",        "1",           nyrs_save),
                    "onset_month":  make_var(onset_month,  "Wet season onset (month)",               "1",           nyrs_save),
                    "onset_year":   make_var(onset_year,   "Wet season onset (year)",                "1",           nyrs_save),
                    "demise_jday":  make_var(demise_jday,  "Wet season demise (Day of Year)",        "day_of_year", nyrs_save),
                    "demise_day":   make_var(demise_day,   "Wet season demise (day of month)",       "1",           nyrs_save),
                    "demise_month": make_var(demise_month, "Wet season demise (month)",              "1",           nyrs_save),
                    "demise_year":  make_var(demise_year,  "Wet season demise (year)",               "1",           nyrs_save),
                    "totwet":       make_var(totwet,       "Accumulated precipitation (wet season)", "mm",          nyrs_save),
                    "totdry":       make_var(totdry,       "Accumulated precipitation (dry season)", "mm",          nyrs_save),
                    "durwet":       make_var(durwet,       "Duration of the wet season",             "days",        nyrs_save),
                    "durdry":       make_var(durdry,       "Duration of the dry season",             "days",        nyrs_save),
                    "smoothed_cycle": xr.DataArray(          # ← inside the dict
                        smoothed,
                        dims=["doy", "lat", "lon"],
                        attrs={"long_name": "Smoothed mean annual cycle (3 harmonics)", "units": "mm/day"},
                    ),
                },
                coords=coords,
                attrs={
                    "model":      model_name,
                    "experiment": exp,
                    "period":     f"{yr_start_coord}-{yr_end_coord}",
                    "method":     "Bombardi et al. (2019) doi:10.1175/BAMS-D-18-0177.1",
                },
            )

            encoding = {v: {"zlib": True, "complevel": 4, "_FillValue": missval}
                        for v in ds_out.data_vars}
            ds_out.to_netcdf(outfile, encoding=encoding)
            print(f"  Saved → {outfile}")
            ds_out.close()

print("\nAll done.")

  SKIP rainy_season_UKESM1-0-LL_HIST_1985-2014.nc — already exists
  SKIP rainy_season_UKESM1-0-LL_SSP245_2021-2050.nc — already exists
  SKIP rainy_season_UKESM1-0-LL_SSP245_2071-2100.nc — already exists
  SKIP rainy_season_UKESM1-0-LL_SSP585_2021-2050.nc — already exists
  SKIP rainy_season_UKESM1-0-LL_SSP585_2071-2100.nc — already exists
  SKIP rainy_season_UKESM1-0-LL_G6solar_2021-2050.nc — already exists
  SKIP rainy_season_UKESM1-0-LL_G6solar_2071-2100.nc — already exists
  SKIP rainy_season_UKESM1-0-LL_G6sulfur_2021-2050.nc — already exists
  SKIP rainy_season_UKESM1-0-LL_G6sulfur_2071-2100.nc — already exists

  CNRM-ESM2-1  HIST  1985-2014
  Calendar class: DatetimeGregorian
  Calendar: DatetimeGregorian  tot=365  ntot=10949  nlat=128  nlon=256
  Step 3 done. Active grid points: 29061/32768
  Step 4: onset/demise loop...
    lat 1/128
    lat 21/128
    lat 41/128
    lat 61/128
    lat 81/128
    lat 101/128
    lat 121/128
  Step 4 done.
  Step 5a done.
  Saved → /gws/ssde/j

In [6]:
###################################### DEBUG ################################################
if debug:
    %store -r cycle_au_Bombardi
    plt.plot(cycle_au, label="Mine: cycle[:,55,69]")
    plt.plot(cycle_au_Bombardi, label="Bombardi: cycle[:,55,69]")
    plt.legend()

In [7]:
###################################### DEBUG ################################################
if debug:
    %store -r prec_au_Bombardi
    plt.plot(prec_au_Mine,     label="Mine: prec[:,55,69]")
    plt.plot(prec_au_Bombardi, label="Bombardi: prec[:,55,69]")
    plt.legend()